In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import librosa
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

In [2]:
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

PyTorch: 2.5.1+cu121
CUDA available: True
Using device: cuda


In [3]:
AUDIO_PATH = '../data/ESC-50-master/ESC-50-master/audio/'
META_PATH = '../data/ESC-50-master/ESC-50-master/meta/esc50.csv'

df = pd.read_csv(META_PATH)

N_MELS = 128
SR = 22050
DURATION = 5
N_FFT = 2048
HOP_LENGTH = 512
NUM_CLASSES = 50

print(f'Total clips: {len(df)}')
print(f'Classes: {df["category"].nunique()}')

Total clips: 2000
Classes: 50


In [4]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['category'])

class BirdSoundDataset(Dataset):
    def __init__(self, df, audio_path):
        self.df = df.reset_index(drop=True)
        self.audio_path = audio_path

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_path = os.path.join(self.audio_path, row['filename'])

        y, sr = librosa.load(file_path, sr=SR, duration=DURATION)

        if len(y) < SR * DURATION:
            y = np.pad(y, (0, SR * DURATION - len(y)))

        S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS,
                                           n_fft=N_FFT, hop_length=HOP_LENGTH)
        S_db = librosa.power_to_db(S, ref=np.max)

        S_db = (S_db - S_db.mean()) / (S_db.std() + 1e-8)

        tensor = torch.tensor(S_db, dtype=torch.float32).unsqueeze(0)

        label = torch.tensor(row['label'], dtype=torch.long)
        return tensor, label

print('Dataset class defined')

Dataset class defined


In [5]:
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label']
)

train_dataset = BirdSoundDataset(train_df, AUDIO_PATH)
test_dataset = BirdSoundDataset(test_df, AUDIO_PATH)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f'Train samples: {len(train_dataset)}')
print(f'Test samples:  {len(test_dataset)}')
print(f'Train batches: {len(train_loader)}')
print(f'Test batches:  {len(test_loader)}')

# verify one batch
batch_x, batch_y = next(iter(train_loader))
print(f'Batch X shape: {batch_x.shape}')
print(f'Batch y shape: {batch_y.shape}')

Train samples: 1600
Test samples:  400
Train batches: 50
Test batches:  13
Batch X shape: torch.Size([32, 1, 128, 216])
Batch y shape: torch.Size([32])


In [6]:
class EchoNet(nn.Module):
    def __init__(self, num_classes=50):
        super(EchoNet, self).__init__()

        self.conv_block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.conv_block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.conv_block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 27, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.classifier(x)
        return x

model = EchoNet(num_classes=NUM_CLASSES).to(device)
print(model)

EchoNet(
  (conv_block1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_block3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=5

In [7]:
with torch.no_grad():
    dummy = torch.zeros(1, 1, 128, 216).to(device)
    out = model.conv_block1(dummy)
    print(f'After block1: {out.shape}')
    out = model.conv_block2(out)
    print(f'After block2: {out.shape}')
    out = model.conv_block3(out)
    print(f'After block3: {out.shape}')
    print(f'Flattened: {out.view(1, -1).shape[1]}')

After block1: torch.Size([1, 32, 64, 108])
After block2: torch.Size([1, 64, 32, 54])
After block3: torch.Size([1, 128, 16, 27])
Flattened: 55296


In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

EPOCHS = 30
train_losses = []
test_accuracies = []

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    scheduler.step()

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            _, predicted = torch.max(outputs, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()

    acc = correct / total
    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)
    test_accuracies.append(acc)

    print(f'Epoch [{epoch+1}/{EPOCHS}] Loss: {avg_loss:.4f} | Test Acc: {acc:.4f}')

Epoch [1/30] Loss: 9.5729 | Test Acc: 0.0600
Epoch [2/30] Loss: 3.7885 | Test Acc: 0.0750
Epoch [3/30] Loss: 3.7254 | Test Acc: 0.1025
Epoch [4/30] Loss: 3.6788 | Test Acc: 0.1425
Epoch [5/30] Loss: 3.5658 | Test Acc: 0.1475
Epoch [6/30] Loss: 3.5390 | Test Acc: 0.1250
Epoch [7/30] Loss: 3.4581 | Test Acc: 0.1875
Epoch [8/30] Loss: 3.3967 | Test Acc: 0.1700
Epoch [9/30] Loss: 3.3835 | Test Acc: 0.1725
Epoch [10/30] Loss: 3.3669 | Test Acc: 0.2100
Epoch [11/30] Loss: 3.2395 | Test Acc: 0.2275
Epoch [12/30] Loss: 3.1391 | Test Acc: 0.2375
Epoch [13/30] Loss: 3.0921 | Test Acc: 0.2125
Epoch [14/30] Loss: 3.0708 | Test Acc: 0.2500
Epoch [15/30] Loss: 3.0324 | Test Acc: 0.2575
Epoch [16/30] Loss: 2.9661 | Test Acc: 0.2625
Epoch [17/30] Loss: 2.9704 | Test Acc: 0.2625
Epoch [18/30] Loss: 2.9163 | Test Acc: 0.2625
Epoch [19/30] Loss: 2.9114 | Test Acc: 0.2625
Epoch [20/30] Loss: 2.9024 | Test Acc: 0.2750
Epoch [21/30] Loss: 2.7767 | Test Acc: 0.2925
Epoch [22/30] Loss: 2.7398 | Test Acc: 0.31

In [9]:
EPOCHS2 = 30

for epoch in range(EPOCHS2):
    model.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    scheduler.step()

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            _, predicted = torch.max(outputs, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()

    acc = correct / total
    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)
    test_accuracies.append(acc)

    print(f'Epoch [{epoch+31}/{60}] Loss: {avg_loss:.4f} | Test Acc: {acc:.4f}')

Epoch [31/60] Loss: 2.4640 | Test Acc: 0.3975
Epoch [32/60] Loss: 2.4463 | Test Acc: 0.3600
Epoch [33/60] Loss: 2.4265 | Test Acc: 0.4000
Epoch [34/60] Loss: 2.3438 | Test Acc: 0.3950
Epoch [35/60] Loss: 2.3546 | Test Acc: 0.4125
Epoch [36/60] Loss: 2.3495 | Test Acc: 0.4050
Epoch [37/60] Loss: 2.3161 | Test Acc: 0.3925
Epoch [38/60] Loss: 2.3993 | Test Acc: 0.3925
Epoch [39/60] Loss: 2.2744 | Test Acc: 0.4325
Epoch [40/60] Loss: 2.2621 | Test Acc: 0.4225
Epoch [41/60] Loss: 2.2904 | Test Acc: 0.4475
Epoch [42/60] Loss: 2.2184 | Test Acc: 0.4275
Epoch [43/60] Loss: 2.1659 | Test Acc: 0.4150
Epoch [44/60] Loss: 2.2160 | Test Acc: 0.4450
Epoch [45/60] Loss: 2.1713 | Test Acc: 0.4525
Epoch [46/60] Loss: 2.1848 | Test Acc: 0.4400
Epoch [47/60] Loss: 2.1442 | Test Acc: 0.4475
Epoch [48/60] Loss: 2.1592 | Test Acc: 0.4525
Epoch [49/60] Loss: 2.1515 | Test Acc: 0.4550
Epoch [50/60] Loss: 2.1520 | Test Acc: 0.4625
Epoch [51/60] Loss: 2.1130 | Test Acc: 0.4375
Epoch [52/60] Loss: 2.0901 | Test 

In [13]:
torch.save(model.state_dict(), '../models/echonet_scratch.pth')
print('Model saved')

Model saved
